**Requesitos**

In [1]:
!pip install transformers datasets peft accelerate torch

  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)


In [2]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

**Carregar Dataset**

In [ ]:
# 1. Carrega o dataset
dataset = load_dataset('json', data_files='dataset_final_divisoes.jsonl')

# 2. Verifica colunas antes da conversão
print("Colunas originais:", dataset["train"].column_names)

# 3. Cria a coluna 'text'
def convert_to_hf_format(example):
    instruction = example.get('Instruction', example.get('instruction', ''))
    output = example.get('Output', example.get('output', ''))
    return {
        "text": f"Instruction: {instruction}\nOutput: {output}"
    }

dataset = dataset.map(convert_to_hf_format)

# 4. Verifica se a coluna foi criada
print("Colunas após conversão:", dataset["train"].column_names)
print("\nPrimeiro exemplo text:")
print(dataset["train"][0]["text"])

# 5. Divide em treino e teste
dataset = dataset["train"].train_test_split(test_size=0.2)
print("\nDataset final:", dataset)

Colunas originais: ['Instruction', 'Output']


Map:   0%|          | 0/167 [00:00<?, ? examples/s]

Colunas após conversão: ['Instruction', 'Output', 'text']

Primeiro exemplo text:
Instruction: Quais são as partes da Alecrim utilizadas e como elas são preparadas para uso medicinal?
Output: A Alecrim é uma planta herbácea, perene, aromática, amplamente cultivada. As partes utilizadas incluem folhas, flores e outros tecidos. A preparação para uso medicinal envolve a infusão das folhas em água fervida, geralmente com uma colher de sobremesa, e a aplicação topical de folhas secas ou óleo de folhas em condições de cólicas infantis, doenças da pele, candidíase e hemorróidas.

Dataset final: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 133
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text'],
        num_rows: 34
    })
})


**Carregar modelo pré-treinado e tokenizador**

In [7]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float32   # float32 para CPU; use bfloat16 se tiver GPU
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Modelo carregado: Qwen/Qwen3-0.6B


**Inferência ANTES do Fine-Tuning**

In [17]:
def generate_response(model, tokenizer, instruction, input_text=""):
    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nOutput:"
    else:
        prompt = f"Instruction: {instruction}\nOutput:"
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    with torch.no_grad():           # ← adicionar isso, economiza memória
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )
    
    # Remove os tokens do prompt da saída
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]          # ← mais limpo para Qwen3
    resposta = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return resposta.strip()

# Exemplo de instrução (deve existir no dataset.jsonl)

test_instruction = "Quais são os usos e formas de preparo do Alecrim?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: Quais são os usos e formas de preparo do Alecrim?
Resposta base: A Alecrim é um medicamento que tem uso comum, mas pode ser preparado de forma medicinal. Ele pode ser preparado com base em árvores, como a árvore de arroz, e pode ser usado


**Tokenização**

In [18]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256 
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

Map:   0%|          | 0/34 [00:00<?, ? examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 133
    })
    test: Dataset({
        features: ['Instruction', 'Output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 34
    })
})
